---
title: "Lab 9: Estadistica Inferencial - No Parametrica"
author: "Maximiliano Garnier Villarreal"
---

# Paquetes

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
from scipy import stats
import pingouin as pg
import statsmodels.stats.api as sms
import statsmodels.api as sm
from statsmodels.formula.api import ols
import plotnine as p9     
import matplotlib.pyplot as plt

p9.theme_set(p9.theme_minimal(base_size = 14))

def signed_rank(x):
    return np.sign(x) * pd.Series(x).rank().values

> Para las pruebas no parametricas se trabaja con los datos **ranqueados**

# Correlacion de Spearman

$$H_0 : \rho_s = 0$$

In [ ]:
a = np.array([8,16,12,13,16,14,16,11,15,13])
b = np.array([7,8,10,12,14,9,13,6,9,10])
c = np.array([3,5,9,5,5,8,13,3,9,9])

a_rank = pd.Series(a).rank().values
b_rank = pd.Series(b).rank().values
c_rank = pd.Series(c).rank().values

alfa = .1

In [ ]:
r = stats.spearmanr(a,b)
print(f"r: {r[0]:.3f}, p: {r[1]:.3f}")

In [ ]:
r = stats.pearsonr(a_rank, b_rank)
r_ci = r.confidence_interval(confidence_level=1-alfa)

print(f"r: {r[0]:.3f}, IC {(1-alfa)*100:.0f}% ({r_ci[0]:.3f}, {r_ci[1]:.3f})")

In [ ]:
pg.corr(a,b,method='spearman')

# Prueba de rango con signo de Wilcoxon

$$H_0 : \text{mediana muestra} = \text{mediana hipotetica}$$

In [ ]:
verm = np.array([6.1, 5.5, 5.3, 6.8, 7.6, 5.3, 6.9, 6.1, 5.7])
alfa = .05
mu = 7

## Prueba

In [ ]:
stats.wilcoxon(verm - mu)

La función `pg.wilcoxon` de **pingouin** realiza la misma prueba y devuelve un resultado similar, incluyendo el tamanho del efecto $r_b$.

In [ ]:
pg.wilcoxon(verm - mu)

## Tamanho del efecto

$$
r = \frac{|Z|}{\sqrt{N}}
$$

$$
r_b = \frac{2 \cdot(w_+ - w_-)}{N \cdot (N+1)}
$$

$$
cles \ (PS) = \frac{2 \cdot w_{max}}{N \cdot (N+1)} 
$$

Funcion para obtener los tamaños de efecto.

In [ ]:
def wilcoxon_effects(x, y=None, mu=0):
    """
    Calcula tamaños de efecto para la prueba de Wilcoxon.
    - Si y is None: one-sample (x - mu)
    - Si y provisto: pareada (x - y)
    Devuelve dict con n, W_plus, W_minus, Z_approx, r, r_b, CLES y scipy_wilcoxon (stat, p).
    """
    x = np.asarray(x)
    if y is None:
        d = x - mu
    else:
        y = np.asarray(y)
        d = x - y

    # eliminar ceros
    mask = (d != 0)
    d = d[mask]
    n = len(d)
    if n == 0:
        raise ValueError("No quedan diferencias (todas cero).")

    ranks = pd.Series(np.abs(d)).rank(method="average").values
    signs = np.sign(d)

    W_plus = ranks[signs > 0].sum()
    W_minus = ranks[signs < 0].sum()
    W_min = min(W_plus, W_minus)

    # esperanza y varianza (sin correcciones por empates)
    expected_W = n * (n + 1) / 4
    var_W = n * (n + 1) * (2 * n + 1) / 24

    # Z aproximado (normal)
    Z = (W_plus - expected_W) / np.sqrt(var_W)

    # tamaños de efecto
    r = abs(Z) / np.sqrt(n)                      # correlación r
    r_b = 2 * (W_plus - W_minus) / (n * (n + 1))  # rank-biserial (pareado / one-sample)
    cles = 2 * max(W_plus, W_minus) / (n * (n + 1))  # CLES (probabilidad de superioridad)

    # resultado de scipy (estadístico W y p-value)
    try:
        scipy_res = stats.wilcoxon(d)
        scipy_stat, scipy_p = float(scipy_res.statistic), float(scipy_res.pvalue)
    except Exception:
        scipy_stat, scipy_p = None, None

    return {
        "n": n,
        "W_plus": W_plus,
        "W_minus": W_minus,
        "W_min": W_min,
        "expected_W": expected_W,
        "var_W": var_W,
        "Z_approx": Z,
        "r": r,
        "r_b (rank_biserial)": r_b,
        "CLES": cles,
        "scipy_wilcoxon_stat": scipy_stat,
        "scipy_wilcoxon_p": scipy_p
    }

In [ ]:
wilcoxon_effects(verm, mu=7)

# Prueba de suma de rangos de Wilcoxon o Prueba-U de Mann-Whitney

$$H_0 : \bar{R}_1 = \bar{R}_2$$

In [ ]:
A = np.array([25, 40, 34, 37, 38, 35, 29, 32, 35, 44, 27, 33, 37, 38, 36])
B = np.array([45, 37, 36, 38, 49, 47, 32, 41, 38, 45, 33, 39, 46, 47, 40])
alfa = .05

data_dict = {
    'A': A,
    'B': B
}

mol = pd.DataFrame({
    'values': np.concatenate([A, B]),
    # 'ind': ['A']*len(A) + ['B']*len(B) 
    'ind': np.concatenate([['A']*len(A), ['B']*len(B)])
})

mol = pd.concat({k: pd.Series(v) for k, v in data_dict.items()}) \
       .reset_index(level=0) \
       .rename(columns={'level_0': 'ind', 0: 'values'}) \
       .reset_index(drop=True)

mol['ranking'] = mol['values'].rank()
mol_res = mol.groupby('ind').agg(n=('values', 'size'),
                              valor_medio=('values', 'mean'),
                              valor_mediana=('values', 'median'),
                              rango_medio=('ranking', 'mean'),
                              w=('ranking', 'sum')).reset_index()
mol_res['u'] = mol_res['w'] - (mol_res['n'] * (mol_res['n'] + 1)) / 2
mol_res

## Prueba

In [ ]:
stats.mannwhitneyu(A, B, alternative='two-sided')

## Tamanho del efecto

$$
r = \frac{|Z|}{\sqrt{N}}
$$

$$
r_b = \frac{2 \cdot (\bar{R}_1-\bar{R}_2)}{N}
$$

$$
\delta = \frac{2 \cdot U}{n_1n_2}-1 
$$

$$
cles \ (PS) = \frac{U_{max}}{n_1n_2} 
$$

La función `pg.mwu` de **pingouin** realiza la misma prueba y devuelve un resultado similar, incluyendo el tamanho del efecto $r_b$ y $cles (PS)$.

In [ ]:
pg.mwu(A, B, alternative='two-sided')

# Prueba de Kruskal-Wallis

$$H_0 : \bar{R}_1 = \bar{R}_2 = \bar{R}_3 = \dotsb = \bar{R}_n$$

In [ ]:
dat1 = pd.read_csv('data/anova MgO.csv')

alfa = .05

dat1['ranking'] = dat1['MgO'].rank()
kw_res = dat1.groupby('Location').agg(n=('MgO', 'size'),
                              valor_medio=('MgO', 'mean'),
                              valor_mediana=('MgO', 'median'),
                              rango_medio=('ranking', 'mean'),
                              w=('ranking', 'sum')).reset_index()
kw_res

## Prueba

In [ ]:
pg.kruskal(dat1, dv='MgO', between='Location')

## Tamanho del efecto

### Epsilon-cuadrado ($\epsilon_{ord}^2$)

$$
\epsilon_{ord}^2 = \frac{H}{(N^2-1)/(N+1)}
$$

In [ ]:
(7.65) / ((12^2 - 1)/(12 + 1))

### Eta-cuadrado ($\eta_H^2$)

$$
\eta_H^2 = \frac{H - k + 1}{N - k}
$$

In [ ]:
(7.65 - 3 + 1) / (12 - 3)

Funcion que calcula los efectos de Kruskal-Wallis.

In [ ]:
def kruskal_effects(data=None, dv=None, group=None, groups_list=None):
    """
    Calcula H (Kruskal-Wallis) y tamaños de efecto epsilon^2 y eta_H^2.
    - data, dv, group: DataFrame y nombres de columnas (dv: variable, group: factor)
    - groups_list: lista de arrays/iterables (alternativa a data)
    Devuelve dict con H, p, N, k, epsilon2, eta_H2.
    """
    if groups_list is None:
        if data is None or dv is None or group is None:
            raise ValueError("Proporcione data+dv+group o groups_list")
        grouped = data.groupby(group)[dv].apply(list)
        groups = [np.asarray(g) for g in grouped.values]
    else:
        groups = [np.asarray(g) for g in groups_list]

    # eliminar NA dentro de cada grupo
    groups = [g[~pd.isna(g)] for g in groups]
    k = len(groups)
    N = sum(len(g) for g in groups)
    if k < 2 or N == 0:
        raise ValueError("Se requieren al menos dos grupos con datos.")

    kr = stats.kruskal(*groups)
    H = float(kr.statistic)
    p = float(kr.pvalue)

    # tamaños de efecto
    epsilon2 = H / ((N**2 - 1) / (N + 1)) if N > 1 else np.nan
    eta_H2 = (H - k + 1) / (N - k) if (N - k) > 0 else np.nan
    if not np.isnan(eta_H2) and eta_H2 < 0:
        eta_H2 = 0.0

    return {
        "H": H,
        "p": p,
        "N": N,
        "k": k,
        "epsilon2": epsilon2,
        "eta_H2": eta_H2
    }

In [ ]:
kruskal_effects(data=dat1, dv='MgO', group='Location')

## Analisis Post-hoc

In [ ]:
import scikit_posthocs as sp

dunn_df = sp.posthoc_dunn(dat1, val_col='MgO', group_col='Location', p_adjust='holm')
dunn_df

# Bootstrap

Remuestreo con remplazamiento, a cada una de la muestras se le calcula el estadistico o dato de interes. El intervalo de confianza es a partir de cuantiles, por lo general 2.5% y 97.5% para un $1 - \alpha = 95 \%$

## Media ($\bar{x}$)

In [ ]:
data_seq = (verm,) # los datos deben ser una secuencia (tupla)
rng = np.random.default_rng(4101)

res = stats.bootstrap(data_seq, 
                      np.mean, 
                      confidence_level=1-alfa, 
                      n_resamples=9999, 
                      method='BCa',
                      rng=rng)

In [ ]:
res_df = pd.DataFrame(res.bootstrap_distribution,columns=['x_bar'])

In [ ]:
print(f"Media: {np.mean(res.bootstrap_distribution):.2f}")

In [ ]:
print(f"Bootstrap IC {(1-alfa)*100:.0f}%: ({res.confidence_interval.low:.4f}, {res.confidence_interval.high:.4f})")

In [ ]:
(p9.ggplot(res_df, p9.aes(x = 'x_bar')) +
  p9.geom_histogram(bins = 20, color = 'black', fill = 'grey') +
  p9.geom_vline(xintercept = np.mean(res.bootstrap_distribution), color = 'blue') +
  p9.geom_vline(xintercept = res.confidence_interval.low, color = 'red') +
  p9.geom_vline(xintercept = res.confidence_interval.high, color = 'red'))

Funcion para bootstrap estratificado

In [ ]:
# python
import numpy as np
import pandas as pd
from typing import Callable, Optional, Dict, Any

def stratified_bootstrap(df: pd.DataFrame,
                         group_col: str,
                         stat_func: Callable[[pd.DataFrame], Any],
                         n_boot: int = 1000,
                         strata_n: Optional[Dict[Any,int]] = None,
                         random_state: Optional[int] = None,
                         ci: float = 0.95):
    """
    Stratified bootstrap.
    - df: DataFrame completo
    - group_col: columna con la etiqueta de estrato
    - stat_func: función que recibe un DataFrame (la muestra completa) y devuelve un escalar o dict de estimadores
    - n_boot: número de remuestreos
    - strata_n: dict opcional {estrato: tamaño_remuestreo}. Si None usa el tamaño observado por estrato.
    - random_state: semilla
    - ci: nivel de confianza (por ejemplo 0.95)
    Devuelve dict con 'distribution' (DataFrame), 'ci' (dict) y 'summary' (DataFrame resumen).
    """
    rng = np.random.default_rng(random_state)
    groups = {name: group.reset_index(drop=True) for name, group in df.groupby(group_col)}
    # determinar tamaños por estrato
    if strata_n is None:
        strata_n = {name: len(g) for name, g in groups.items()}

    results = []
    for _ in range(n_boot):
        parts = []
        for name, g in groups.items():
            n = int(strata_n.get(name, len(g)))
            if n == 0:
                continue
            idx = rng.integers(0, len(g), size=n)
            parts.append(g.iloc[idx])
        sample = pd.concat(parts, ignore_index=True)
        res = stat_func(sample)
        results.append(res)

    # normalizar salida a DataFrame
    if len(results) == 0:
        raise ValueError("No hay resultados de bootstrap.")
    # si stat_func devuelve escalares numéricos
    if np.isscalar(results[0]):
        arr = np.array(results)
        dist_df = pd.DataFrame({"stat": arr})
        lower = np.percentile(arr, (1 - ci)/2*100)
        upper = np.percentile(arr, (1 + ci)/2*100)
        ci_dict = {"stat": (lower, upper)}
        summary = pd.DataFrame({
            "estimate": [arr.mean()],
            "std_error": [arr.std(ddof=1)],
            f"ci_lower_{ci:.2f}": [lower],
            f"ci_upper_{ci:.2f}": [upper]
        })
    else:
        # si devuelve dicts o múltiples estadísticas
        # convertir lista de dict -> DataFrame
        dist_df = pd.DataFrame(results)
        ci_dict = {}
        rows = []
        for col in dist_df.columns:
            arr = dist_df[col].dropna().to_numpy()
            lower = np.percentile(arr, (1 - ci)/2*100)
            upper = np.percentile(arr, (1 + ci)/2*100)
            ci_dict[col] = (lower, upper)
            rows.append({
                "stat": col,
                "estimate": arr.mean(),
                "std_error": arr.std(ddof=1),
                f"ci_lower_{ci:.2f}": lower,
                f"ci_upper_{ci:.2f}": upper
            })
        summary = pd.DataFrame(rows).set_index("stat")

    return {
        "distribution": dist_df,
        "ci": ci_dict,
        "summary": summary
    }

## $\eta^2$

In [ ]:
def eta(df):
  res = pg.kruskal(df, dv='MgO', between='Location')
  H = res['H'][0]
  N = len(df)
  k = res['ddof1'][0]
  eta = (H - k + 1) / (N - k)
  return eta

In [ ]:
alfa = .05

res = stratified_bootstrap(df=dat1, group_col='Location', 
                           stat_func=eta, ci=1-alfa,
                           n_boot=10000, random_state=4101)

In [ ]:
res["summary"]

In [ ]:
(p9.ggplot(res["distribution"], p9.aes(x = 'stat')) +
  p9.geom_histogram(bins = 20, color = 'black', fill = 'grey') +
  p9.geom_vline(xintercept = res["summary"]['estimate'][0], color = 'blue') +
  p9.geom_vline(xintercept = res['ci']['stat'][0], color = 'red') +
  p9.geom_vline(xintercept = res['ci']['stat'][1], color = 'red'))